# GraphQueryPort command line
`python -m knowledge_palace.graph.port <operation>` prints one envelope as JSON. The fixture Vault and a temporary `.palace.toml` stand in for a library.

In [ ]:
import json, shutil, subprocess, sys, tempfile
from pathlib import Path
root = Path.cwd()
while not (root / "knowledge_palace").is_dir():
    root = root.parent
sys.path.insert(0, str(root))
from knowledge_palace.graph.builder import write_index
from knowledge_palace.tests.test_research import MINI
temp = tempfile.TemporaryDirectory()
base = Path(temp.name)
shutil.copytree(MINI, base / "vault")
for name in ("state", "source", "workspace"):
    (base / name).mkdir()
config = base / ".palace.toml"
config.write_text("".join('%s_dir = "%s"\n' % (name, name) for name in ("vault", "state", "source", "workspace")), encoding="utf-8")
write_index(base / "vault", base / "state")

def port(*arguments):
    run = subprocess.run([sys.executable, "-m", "knowledge_palace.graph.port", "--config", str(config), *arguments],
                         cwd=root, capture_output=True, text=True, encoding="utf-8", check=True)
    return json.loads(run.stdout)

hierarchy = port("list_hierarchies")["hierarchies"][0]["id"]
hierarchy, [level["label"] for level in port("list_levels", hierarchy)["levels"]]

In [ ]:
card = port("get_content", "--", "work:alpha-2020-echo")
context = port("query_context", "claim:alpha-2020-echo#C1")
page = port("query_level", hierarchy, "2", "--limit", "2")["page"]
missing = port("get_content", "--", "work:no-such-card")
assert card["canonical_ref"]["path"] == "papers/alpha-2020-echo.md" and "## Claims" in card["content"]
assert [view["id"] for view in context["parents"]] == ["work:alpha-2020-echo"]
assert page["returned"] == 2 and page["truncated"]
assert missing["error"]["code"] == "unknown_node"  # a typed error is still exit 0 and JSON
card["snapshot"]["snapshot_id"], context["node"]["attrs"], page["next_cursor"], missing["error"]

In [ ]:
temp.cleanup()